In [47]:
from math import *
import numpy as np

import pandas as pd
import requests
import io

from astropy.io import fits
from astropy.wcs import WCS
from astropy.time import Time
from astropy.time import TimeDelta

from scipy.optimize import curve_fit

from astropy import units as u
from astropy.coordinates import SkyCoord

import glob
import os
import re

import matplotlib.pyplot as plt
import matplotlib
from matplotlib import cm
%matplotlib widget

In [48]:
upath = 'fits/'
filelist = sorted(glob.glob(f'{upath}Uranus*wcs.fits'))
print('{}'.format(filelist))

['fits/Uranus_R_4_0001_wcs.fits', 'fits/Uranus_R_4_0002_wcs.fits', 'fits/Uranus_R_4_0003_wcs.fits', 'fits/Uranus_R_4_0004_wcs.fits', 'fits/Uranus_R_4_0005_wcs.fits']


In [49]:
hdul = fits.open(filelist[4])
img = (hdul[0].data).astype(float)
h, w = np.shape(img)
hdr = hdul[0].header
hdr

SIMPLE  =                    T / conforms to FITS standard                      
BITPIX  =                   32 / array data type                                
NAXIS   =                    2 / number of array dimensions                     
NAXIS1  =                 2072                                                  
NAXIS2  =                 1410                                                  
EXTEND  =                    T                                                  
TARGET  = 'Uranus  '                                                            
DATE-OBS= '2025-09-21T22:42:59.873596'                                          
DATE-END= '2025-09-21T22:43:10.365809'                                          
RA      = '03:55:37.2'                                                          
DEC     = '+20:08:58.4'                                                         
EXPTIME =                 10.0                                                  
FOCUS   =                 62

In [50]:
def angular_distance(ra,dec,RA,DEC):
    return acos(sin(dec)*sin(DEC)+cos(dec)*cos(DEC)*cos(ra-RA))

def equatorial_to_tangential(ra,dec,RA,DEC):
    ksi = cos(dec)*sin(ra-RA)/(sin(dec)*sin(DEC)+cos(dec)*cos(DEC)*cos(ra-RA))
    eta = (sin(dec)*cos(DEC)-cos(dec)*sin(DEC)*cos(ra-RA))/(sin(dec)*sin(DEC)+cos(dec)*cos(DEC)*cos(ra-RA))
    return ksi,eta

def tangential_to_equatorial(ksi,eta,RA,Dec):
    x,y,z = np.dot(np.array([[-sin(RA),-cos(RA) * sin(Dec),cos(RA) * cos(Dec)],
                             [cos(RA),-sin(RA) * sin(Dec),sin(RA) * cos(Dec)],
                             [0,cos(Dec),sin(Dec)]]),
                   np.array([ksi,eta,1]))/sqrt(1+ksi*ksi+eta*eta)
    ra = atan2(y,x)
    dec = atan2(z,sqrt(x*x+y*y))
    if(ra<0):
        ra+=2*pi
    return ra,dec

In [51]:
### Получаем ra, dec, gmag для звёзд в окрестности кадра

In [52]:
ra0, dec0 = float(hdr["CRVAL1"]), float(hdr["CRVAL2"])
ra0, dec0

(58.9159857586, 20.1538380712)

In [53]:
t = Time(hdr['DATE-OBS'], format='isot')
t+=TimeDelta(float(hdr['EXPTIME']), format='sec')/2
JD = float(t.copy(format='jd').value)
yr = float(t.copy(format='jyear').value) 


fov = 0.4

JD, fov

(2460940.446584185, 0.4)

In [54]:
def get_Gaia_data(yr, lm1,lm2, gdf):
    ra,dec,gmag = [],[],[]
    for index, row in gdf.iterrows():
        G = row['phot_g_mean_mag']
        if (not isnan(G)) and (lm1<G<lm2):
            ra0,dec0,pmra,pmdec,ep = \
                radians(row['ra']), \
                radians(row['dec']),\
                radians(row['pmra']/3600000.0),\
                radians(row['pmdec']/3600000.0),\
                row['ref_epoch']
            if not isnan(pmra):
                alpha,delta = np.degrees(tangential_to_equatorial(pmra*(yr-ep),pmdec*(yr-ep),ra0,dec0))
                ra.append(alpha)
                dec.append(delta)
                gmag.append(G)
    return np.array(ra),np.array(dec),np.array(gmag)

In [55]:
response = requests.get(f'http://sfa.puldb.ru:9810?cmd=box&ra={ra0}&dec={dec0}&fov={fov}')
df = pd.read_csv(io.StringIO(response.content.decode('utf-8')))
df

,solution_id,designation,source_id,random_index,ref_epoch,ra,ra_error,dec,dec_error,parallax,...,azero_gspphot,azero_gspphot_lower,azero_gspphot_upper,ag_gspphot,ag_gspphot_lower,ag_gspphot_upper,ebpminrp_gspphot,ebpminrp_gspphot_lower,ebpminrp_gspphot_upper,libname_gspphot
0,1636148068921376768,Gaia DR3 51258373093921280,51258373093921280,345028988,2016.0,59.125298,0.216448,19.957065,0.137909,0.021883,...,0.0148,0.0033,0.0432,0.0112,0.0025,0.0326,0.0060,0.0013,0.0176,MARCS
1,1636148068921376768,Gaia DR3 51258510532872192,51258510532872192,400992464,2016.0,59.082375,0.369506,19.953836,0.234851,1.017526,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1636148068921376768,Gaia DR3 51258544892612096,51258544892612096,1382546787,2016.0,59.106924,0.133514,19.955876,0.104925,1.340692,...,0.5511,0.5178,0.5837,0.3811,0.3580,0.4038,0.2094,0.1969,0.2216,PHOENIX
3,1636148068921376768,Gaia DR3 51258549189666176,51258549189666176,1229386944,2016.0,59.110723,0.106526,19.959347,0.075979,0.357447,...,0.0329,0.0071,0.0939,0.0253,0.0054,0.0726,0.0135,0.0029,0.0389,PHOENIX
4,1636148068921376768,Gaia DR3 51258583549403008,51258583549403008,1224435394,2016.0,59.114679,0.040356,19.963272,0.031519,0.425562,...,0.8491,0.8048,0.8709,0.6908,0.6540,0.7091,0.3726,0.3528,0.3824,PHOENIX
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1196,1636148068921376768,Gaia DR3 51353107189730944,51353107189730944,1439971659,2016.0,58.839558,0.052091,20.350220,0.033324,0.507913,...,0.0653,0.0239,0.2034,0.0517,0.0189,0.1621,0.0279,0.0102,0.0871,PHOENIX
1197,1636148068921376768,Gaia DR3 51353240331884416,51353240331884416,840026978,2016.0,58.883346,0.254405,20.350842,0.175563,0.439484,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1198,1636148068921376768,Gaia DR3 51353244627240960,51353244627240960,1023939329,2016.0,58.882036,0.894397,20.352231,0.680267,0.744170,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1199,1636148068921376768,Gaia DR3 51353790087696640,51353790087696640,322811323,2016.0,58.977139,0.744305,20.347207,0.516746,-0.337609,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [56]:
limmag1,limmag2 = 6.0,16.0
ra, dec, gmag = get_Gaia_data(yr,limmag1,limmag2, df)
len(ra)

90

In [74]:
transform_mtx = np.array([[ hdr['CD1_1'], hdr['CD1_2'] ], \
                          [ hdr['CD2_1'], hdr['CD2_2'] ]])

x0, y0 = hdr['CRPIX1'], hdr['CRPIX2']
                          
def tangential_to_pixel(xi, eta, transform=transform_mtx, x0=x0, y0=y0):
    x, y = np.dot(np.linalg.inv(transform), [xi, eta]) + np.array([x0, y0])
    return x, y

def pixel_to_tangential(x, y, transform=transform_mtx, x0=x0, y0=y0):
    xi, eta = np.dot(transform, [x-x0, y-y0])
    return xi, eta

In [58]:
eq2pix = lambda ra, dec: tangential_to_pixel(*np.rad2deg( \
                            equatorial_to_tangential( *np.deg2rad((ra, dec, ra0, dec0)) ) ))

In [59]:
### Фильтруем звёзды

In [60]:
W = WCS(hdul[0].header)

Set MJD-END to 60939.946648 from DATE-END'. [astropy.wcs.wcs]


In [67]:
bdr = 50
pxpos = W.wcs_world2pix(np.stack((ra, dec), axis=-1),1)  
p_indx = np.where((pxpos[:,0]>bdr) & (pxpos[:,1]>bdr) & (pxpos[:,0]<w-bdr) & (pxpos[:,1]<h-bdr))
pixpos = pxpos[p_indx]
ra,dec,gmag = ra[p_indx],dec[p_indx],gmag[p_indx]

x = [p[0] for p in pxpos]
y = [p[1] for p in pxpos]

In [68]:
### Отмечаем на кадре

In [69]:
from astropy.stats import sigma_clip

img_processed = sigma_clip(img, sigma_lower=10.0, sigma_upper=10.0) # Возводим в квадрат для увеличения контраста

fig, ax = plt.subplots(figsize=(3.0,3.0), dpi=300)
ax.xaxis.set_tick_params(labelsize=5)
ax.yaxis.set_tick_params(labelsize=5)
plt.tight_layout()

ax.imshow(img_processed, origin='lower', cmap="gray") 
plt.show()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [70]:
size = 20
line_width = 0.2
ax.scatter(x, y, marker='o', facecolors='none', edgecolors='red', s=size, lw=line_width)

In [71]:
### Проводим регрессию координат

In [72]:
from scipy.optimize import curve_fit

In [96]:
def gaussian2D(xy,*p):
    x, y = xy
    Ibkg,Imax,A,B,C,xc,yc = p
    r = A*(x-xc)**2+C*(y-yc)**2+B*(x-xc)*(y-yc)
    return Ibkg+Imax*np.exp(-r)

def im_centroiding(I):
    x_c = int(I.shape[0]/2)
    y_c = int(I.shape[0]/2)
    Ib, Im = np.median(I),np.max(I) 
    A = C = 1/5
    B = 0
    xx, yy = np.meshgrid(np.arange(x_c), np.arange(y_c))
    return curve_fit(f=gaussian2D, p0=np.array([Ib,Im,A,B,C,x_c,y_c]),\
                     xdata = (xx, yy),ydata=I.ravel(),maxfev=1000)

In [97]:
ap = 20
xp,yp = np.array([]),np.array([])
xi,eta = np.array([]),np.array([])

for px, py in (pxpos).astype(int):
    I = img[px-ap:py+ap,px-ap:px+ap]
    #try:
    p, covp = im_centroiding(I)
    #except:
    #    print((px,py), "Regression failure")
    #    continue
    Ibkg,Imax,A,B,C,xc,yc = p
    if sqrt(covp[5,5])<0.25 and sqrt(covp[6,6])<0.25:
        xp,yp = np.append(xp,x-ap+xc),np.append(yp,y-ap+yc)
        # добавить свою функцию для вычисления тангенциальных координат
        tx,ty = pixel_to_tangential(xp, yp)
        np.append(xi,tx)
        np.append(eta,ty)

ValueError: operands could not be broadcast together with shapes (1157776,) (43040,) 

In [84]:
xi

array([], dtype=float64)